In [5]:
!pip install pandas

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.3 MB 8.3 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/11.3 MB 2.6 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/11.3 MB 2.2 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/11.3 MB 2.0 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/11.3 MB 1.8 MB/s eta 0:00:06
   ------- -------------------------------- 2.1/11.3 MB 1.7 MB/s eta 0:00:06
   -------- ------------------------------- 2.4/11.3 MB 1.6 MB/s eta 0:00:06
   --------- ------------------------------ 2.6/11.3 MB 1.6 MB/s eta 0:00:06
   ---------- ----------------------------- 2.9/11.3 MB 1.5 MB/s eta 0:00:06
   ----------- ---------------------------- 3.1/11.3 MB 1.5 MB/s eta 0:00:06
   ------------ --------------------------- 3.4/11.3 MB 1.5 MB/s eta 0:00:06
   ------------ --------------------------- 3.7/11.3 MB 1.5 MB/s eta 0:00:06
   ---

In [7]:
# 데이터 로드
import pandas as pd

df = pd.read_csv('ragbase.csv', encoding='cp949')

# 데이터 확인
display(df.head(3))

,cause,name,url,target,action_name,action_des,source,takes,time,need,level,saving
0,냉방설비 노후화,한전 소상공인 고효율기기 지원사업,https://online.kepco.co.kr/,소상공인,1등급 냉난방기 교체,노후 기기를 1등급 고효율 가전으로 교체하여 전력 소비 대폭 절감,한국전력공사,기기 구매가격(부가세 제외)의 최대 40% (한도 160만원),2026 .02 .09 ~ 예산 소진 시,"중소기업(소상공인)확인서, 사업자등록증, 기기명판/라벨, 영수증, 전경사진",중간,연간 약 100~150kg 절감 (1대 기준)
1,냉방설비 노후화,소상공인 에너지효율향상 지원사업,https://www.energy.or.kr/,소상공인,노후 냉방기 교체,에너지소비효율 1등급 제품 교체 지원으로 냉방비 근본적 절감,한국에너지공단,품목당 한도 내 구매 비용의 40% 지원,2026 .02 .10 ~ 2026 .12 .31,"소상공인확인서, 신분증, 제품/라벨 사진, 구매증빙",중간,연간 약 100~150kg 절감 (1대 기준)
2,냉방설비 노후화,소공인 클린제조환경조성사업,https://www.sbiz24.kr,소상공인,작업장 설비 개선,소공인 작업장 내 노후화된 냉난방 및 환기 설비를 고효율로 교체,"중소벤처기업부, 소상공인시장진흥공단",업체당 최대 490만원 내외 (국비 70%),2026 .04 .30 ~ 2026 .06 .02,"소공인확인서, 사업자등록증, 국세/지방세 완납증명서, 견적서",어려움,기존 설비 대비 탄소 배출 약 20~30% 절감


In [9]:
# 템플릿 매핑용 데이터 추출 함수
def get_recommendation_data(cause: str, target: str) -> dict:
    # 1. 하드 필터링: 원인과 타겟이 일치하는 데이터만 추출
    filtered_df = df[(df['cause'] == cause) & (df['target'].str.contains(target))]
    
    # 2. 결과가 없을 경우의 응답 템플릿 포맷
    if filtered_df.empty:
        return {
            "status": "empty",
            "message": "현재 조건에 딱 맞는 지원사업이 없습니다.",
            "data": []
        }
    
    # 3. 딕셔너리 리스트로 변환
    business_list = []
    for _, row in filtered_df.iterrows():
        business = {
            "title": row['name'],                      # UI의 카드 제목
            "action_title": row['action_name'],        # 추천 액션명
            "description": row['action_des'],          # 상세 설명
            "benefit": row['takes'],                   # 지원 규모 
            "documents": row['need'],                  # 필요 서류
            "link": row['url'],                        # 자세히 보기 버튼 링크
            "difficulty": row['level'],                # 난이도 라벨
            "carbon_saving": row['carbon_saving_yr']   # 연간 탄소 절감량
        }
        business_list.append(business)
        
    # 4. 최종 응답 데이터 구성
    return {
        "status": "success",
        "user_cause": cause,
        "user_target": target,
        "total_count": len(business_list),
        "data": business_list
    }